In [ ]:
'''
Assignment 1
Student Name: Aaron Baggot  
Student Number: C22716399 
Module: IDSP1701
TU856/4  

Parameters description:

ROTATE: True rotates to horizontal with a rotated bounding box, False uses a plain axis-aligned crop.
CLAHE_CLIP = 3.0, how strongly CLAHE boosts local contrast between 2.0 and 4.0 best results.
KSIZE = 5, morphology kernel (odd, 5 or 7 common).
CLOSE_IT = 1, how many times to run closing (fills holes).
DILATE_IT = 2, grows the mask slightly to avoid fragmented or thin foreground.
PAD_FRAC = 0.15, extra padding around the crop (as a fraction of min image size).If the tails or haed get clipped it's increased [1].

This assignment builds on the functions covered in the labs using Python and OpenCV [2]. 
Tested out a few different approaches, and the one that worked best followed these main steps:

Bilateral Filter:

The bilateral filter helps reduce speckle and patchy noise while keeping the edges sharp. 
It does this by weighting pixels based on both how close they are in space and how similar their intensity is. 
Underwater images can be really grainy and uneven, so this step smooths out the water without making the shark's outline blurry [3].  

Example code: 

    den = cv.bilateralFilter(I, 7, 75, 75)

CLAHE on the LAB L Channel:

Next, applied CLAHE (Contrast Limited Adaptive Histogram Equalisation) to the L (lightness) channel in the LAB color space. 
This boosts the local contrast in different parts of the image while preventing noise from getting too amplified 
by clipping the big peaks. 
It really helps with low-contrast underwater scenes and brings out details like the shark's body stripes [4].  

Example code: 

    L2 = cv.createCLAHE(clipLimit = CLAHE_CLIP, tileGridSize = (8,8)).apply(L)

Light Unsharp Mask:

A small Gaussian blur and a weighted addition was used to gently sharpen the edges and bring back some definition 
that might have been lost in the denoising and contrast steps.This keeps the image looking natural. 
The Gaussian blur gives more weight to pixels closer to the center of the kernel [5].  

Example code:  

    blur = cv.GaussianBlur(enh, (0,0), 1.0)
    enh = cv.addWeighted(enh, 1.20, blur, -0.20, 0)


Otsu Thresholding for Segmentation:

For segmentation, Otsu's method was implemented, which automatically picks the best threshold for turning the image into black and white. 
Two simple versions were created so the same code can handle different types of images.  

    • Underwater mode: Otsu on the LAB-b channel (blue-yellow) combined with the inverted HSV saturation 
                       to separate the fish from the open water.  
    • Silhouette mode: Otsu on the inverted grayscale for darker, low-saturation shapes like Shark 2 image.  

This is implemented using `cv.threshold`  `THRESH_OTSU` [6].  

Example code: 

    _, mask = cv.threshold(inv, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)

Morphology and Largest Component:

Then applied some morphological operations, closing to fill small holes, opening to remove specks, 
and a light dilation to thicken thin parts of the mask. 
This gives a clean blob for cropping without needing much tweaking [7]. 
After that, labeled the connected components and kept only the largest one to get rid of any stray pixels or bubbles [8].  

Example code:  

    k = cv.getStructuringElement(cv.MORPH_ELLIPSE, (KSIZE, KSIZE))
    mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, k, iterations = CLOSE_IT)
    num, labc = cv.connectedComponents(mask)


Rotate and Crop:

For underwater images, I straightened the mask's contour using a minimum-area rectangle and 
rotated it to be horizontal with "cv.minAreaRect` and `cv.getRotationMatrix2D". 
For regular photos, like the cat in snow, simple axis-aligned bounding box to avoid weird wedge effects [9], 
and added a bit of padding so the subject doesn't get cut off [10].  

Example code:  

    rect = cv.minAreaRect(cnt)
    M = cv.getRotationMatrix2D((W//2, H//2), rect[-1], 1.0)
    I_rot = cv.warpAffine(enh, M, (W, H))


Minimal Background:

Finally, sampled pixels from a thin border outside the mask, took the median color for each channel, 
and used that as a flat background. This creates the clean, minimal background needed [11].  

Summary

Overall, the program takes an image with a plain background and outputs an enhanced version that's clearer and centered on a solid background. 
The bilateral filter cuts down on noise while keeping edges intact. 
Otsu thresholding makes a foreground mask using color info for blue underwater shots or inverted grayscale for silhouettes. 
Small morphological ops clean up the mask, and keep the biggest blob. 
Then, the shark gets rotated to horizontal for underwater pics or cat cropped for normal ones, 
and placed on a background color guessed from the image's border. The whole pipeline is short and only uses a few adjustable values.

References

[1] "pypfilt.plot — pypfilt 0.8.7 documentation." Accessed: Oct. 16, 2025. [Online]. Available: https://pypfilt.readthedocs.io/en/latest/api/plot.html

[2] J. Howse, P. Joshi, and M. Beyeler, OpenCV: computer vision projects with Python, 1st ed. Packt Publishing, 2016.  

[3] C. Tomasi and R. Manduchi, "Bilateral filtering for gray and color images," in Sixth International Conference on Computer Vision*, Jan. 1998, pp. 839–846. doi: 10.1109/ICCV.1998.710815.  

[4] J. Wu, "Enhancing Object Sorting Under Low-Light Conditions with CLAHE, Gaussian Blur, ROI, and Custom PID on a Raspberry Pi Robotic Arm," Appl. Comput. Eng., vol. 96, no. 1, pp. 93–98, Nov. 2024, doi: 10.54254/2755-2721/96/20241240.  

[5] S. Singh, R. Verma, and A. K. Singh, "Image filtration in Python using openCV," Turk. J. Comput. Math. Educ., vol. 12, no. 6, pp. 5136–5143, 2021.  

[6] M. Huang, W. Yu, and D. Zhu, "An Improved Image Segmentation Algorithm Based on the Otsu Method," in 2012 13th ACIS International Conference on Software Engineering, Artificial Intelligence, Networking and Parallel/Distributed Computing, Aug. 2012, pp. 135–139. doi: 10.1109/SNPD.2012.26.  

[7] J. Jenrette, Z. Y.-C. Liu, P. Chimote, T. Hastie, E. Fox, and F. Ferretti, "Shark detection and classification with machine learning," Ecol. Inform., vol. 69, p. 101673, July 2022, doi: 10.1016/j.ecoinf.2022.101673.  

[8] F. Bolelli, S. Allegretti, and C. Grana, "Quest for Speed: The Epic Saga of Record-Breaking on OpenCV Connected Components Extraction".  

[9] S. Rath, "Contour Detection using OpenCV (Python/C++)." Accessed: Oct. 19, 2025. [Online]. Available: https://learnopencv.com/contour-detection-using-opencv-python-c

[10] A. Mordvintsev, "OpenCV-Python Tutorials Documentation".  

[11] A. P. Athane and D. S. R. Prasad, “Image Enhancement Based on Opencv Using Python 2.7 – Review,” vol. 8, no. 5, 2021.


'''


In [3]:
from pathlib import Path
import cv2 as cv
import numpy as np
import os
import matplotlib.pyplot as plt

# PARAMS
INPUT_PATH = Path("/Users/aaronbaggot/Desktop/Image_Processing_Assignment_1/data/raw/Shark 1.PNG")   # change for Shark 2 or Shark 1
MODE   = "underwater"    # "underwater" or "silhouette"
ROTATE = True            # False for cat or diferent image
CLAHE_CLIP = 3.0
KSIZE      = 5
CLOSE_IT   = 1
DILATE_IT  = 2
PAD_FRAC   = 0.15



def enhance_image(I):
    """Enhance the image using bilateral filter, CLAHE, and sharpening."""
    den = cv.bilateralFilter(I, 7, 75, 75)
    L, A, B = cv.split(cv.cvtColor(den, cv.COLOR_BGR2LAB))
    L2 = cv.createCLAHE(clipLimit = CLAHE_CLIP, tileGridSize = (8,8)).apply(L)
    enh = cv.cvtColor(cv.merge([L2, A, B]), cv.COLOR_LAB2BGR)
    blur = cv.GaussianBlur(enh, (0,0), 1.0)
    enh = cv.addWeighted(enh, 1.20, blur, -0.20, 0)
    return enh

def segment_image(enh, mode):
    """Segment the image to create a mask based on mode."""
    if mode == "underwater":
        b = cv.cvtColor(enh, cv.COLOR_BGR2LAB)[:, :, 2]
        S = cv.cvtColor(enh, cv.COLOR_BGR2HSV)[:, :, 1]
        _, th_b = cv.threshold(b, 0, 255, cv.THRESH_BINARY+cv.THRESH_OTSU)
        _, s_inv = cv.threshold(255 - S, 0, 255, cv.THRESH_BINARY+cv.THRESH_OTSU)
        mask = cv.bitwise_and(th_b, s_inv)
        if (mask.sum()/255)/mask.size < 0.03:
            mask = cv.bitwise_or(th_b, s_inv)
    else:
        g = cv.cvtColor(enh, cv.COLOR_BGR2GRAY)
        gb = cv.GaussianBlur(g, (5,5), 0)
        inv = cv.normalize(255 - gb, None, 0, 255, cv.NORM_MINMAX)
        _, mask = cv.threshold(inv, 0, 255, cv.THRESH_BINARY+cv.THRESH_OTSU)
    return mask


def morph_mask(mask, ksize, close_it, dilate_it):
    """Apply morphological operations to the mask."""
    k = cv.getStructuringElement(cv.MORPH_ELLIPSE, (ksize, ksize))
    mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, k, iterations=close_it)
    mask = cv.morphologyEx(mask, cv.MORPH_OPEN, k, iterations=1)
    if dilate_it > 0:
        mask = cv.dilate(mask, k, iterations=dilate_it)
    
    # Select largest component
    num, labc = cv.connectedComponents(mask)
    if num > 1:
        areas = [(labc == i).sum() for i in range(1, num)]
        mask = (labc == (1 + int(np.argmax(areas)))).astype(np.uint8) * 255
    
    return mask

def rotate_and_crop(enh, mask, rotate, pad_frac):
    """Rotate the image and mask if needed, then crop to bounding box with padding."""
    H, W = enh.shape[:2]
    if rotate:
        cnts, _ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
        cnt = max(cnts, key=cv.contourArea)
        rect = cv.minAreaRect(cnt); angle = rect[-1]; (rw, rh) = rect[1]
        if rw < rh: angle += 90
        if angle < -45: angle += 90
        M = cv.getRotationMatrix2D((W//2, H//2), angle, 1.0)
        I_r = cv.warpAffine(enh, M, (W, H), flags = cv.INTER_LINEAR, borderMode = cv.BORDER_REFLECT)
        m_r = cv.warpAffine(mask, M, (W, H), flags = cv.INTER_NEAREST, borderMode = cv.BORDER_CONSTANT)
        cnt_t = cv.transform(cnt.reshape(-1,1,2).astype(np.float32), M).astype(np.int32)
        x, y, w, h = cv.boundingRect(cnt_t)
    else:
        cnts, _ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
        x, y, w, h = cv.boundingRect(max(cnts, key = cv.contourArea))
        I_r, m_r = enh, mask
    
    pad = int(pad_frac * min(W, H))
    x0 = max(0, x - pad); y0 = max(0, y - pad)
    x1 = min(W, x + w + pad); y1 = min(H, y + h + pad)
    crop = I_r[y0:y1, x0:x1]
    mcrp = m_r[y0:y1, x0:x1]
    return crop, mcrp

def replace_background(crop, mcrp):
    """Replace background with median color from border pixels."""
    inv = (mcrp == 0).astype(np.uint8) * 255
    border = np.zeros_like(inv)
    border[:10, :] = 255; border[-10:, :] = 255
    border[:, :10] = 255; border[:, -10:] = 255
    samples = cv.bitwise_and(crop, crop, mask=cv.bitwise_and(inv, border)).reshape(-1, 3)
    samples = samples[(samples.sum(1) > 0)]
    bg = (210, 220, 230) if len(samples) == 0 else tuple(map(int, np.median(samples, 0)))
    canvas = np.full_like(crop, bg, dtype = np.uint8)
    final = cv.add(cv.bitwise_and(crop, crop, mask = mcrp),
                   cv.bitwise_and(canvas, canvas, mask = cv.bitwise_not(mcrp)))
    return final


def process_image(inp: Path):
    """Main processing function using modular steps."""
    I = cv.imread(str(inp), cv.IMREAD_COLOR)
    if I is None:
        raise FileNotFoundError(inp)
    
    enh = enhance_image(I)
    mask = segment_image(enh, MODE)
    mask = morph_mask(mask, KSIZE, CLOSE_IT, DILATE_IT)
    
    area = (mask.sum() / 255) / mask.size
    if area < 0.005 or area > 0.98:
        return enh
    
    crop, mcrp = rotate_and_crop(enh, mask, ROTATE, PAD_FRAC)
    final = replace_background(crop, mcrp)
    return final

   
# Run the processing
final_img = process_image(INPUT_PATH)
plt.imshow(cv.cvtColor(final_img, cv.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Output")
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

# Troubleshooting git push

Quick checks to run if `git push` fails:
- Read the exact git error message first.
- Run `git status` to ensure all changes are committed.
- Check remote and branch: `git remote -v` and `git branch -vv`.
- Confirm authentication (SSH key or PAT) and repository permissions.
- Look for large files (Git LFS) or pre-push hooks blocking push.
- Use `git push --set-upstream origin <branch>` if branch not tracked.

In [ ]:
# Git diagnostic commands — run from notebook (note: prepend with ! to run shell commands)
!git status
!git remote -v
!git branch -vv
!git log -1 --pretty=%B
# Try to push (replace <branch> if needed):
!git push --dry-run origin HEAD || true
# If branch not tracked, set upstream (uncomment and replace <branch>):
# !git push --set-upstream origin <branch>
# Check for large files or LFS issues:
!git lfs env || echo 'git lfs not installed or no repo LFS'
# To test authentication quickly:
!git ls-remote origin || echo 'ls-remote failed — check auth and remote URL'

## Ensure VS Code uses the arm64 dotnet host

1. Add the dotnet host folder to your shell PATH (zsh):
   - This makes the correct arm64 dotnet available to VS Code when launched from the terminal or when VS Code reads your shell environment.
2. (Optional) Add the same PATH prefix to VS Code user settings so integrated terminals start with the same PATH.
3. Restart VS Code after changes and verify with `which dotnet` and `dotnet --info` in the integrated terminal.

In [ ]:
# Replace <USER_HOME> with your home if needed — these commands use your exact path below.
# 1) Add to your zsh profile (~/.zshrc). This prepends the folder containing 'dotnet'.
DOTNET_DIR="/Users/aaronbaggot/Library/Application Support/Code/User/globalStorage/ms-dotnettools.vscode-dotnet-runtime/.dotnet/9.0.11~arm64"
echo "export PATH=\"$DOTNET_DIR:$PATH\"" >> ~/.zshrc
echo "# Added arm64 dotnet host for VS Code extension" >> ~/.zshrc
echo "Run: source ~/.zshrc or restart terminal" 

# 2) Optional: add to VS Code user settings (open and paste the snippet below into ~/Library/Application Support/Code/User/settings.json)
cat <<'JSON_SNIPPET'
{
  // ...other settings...
  "terminal.integrated.env.osx": {
    "PATH": "/Users/aaronbaggot/Library/Application Support/Code/User/globalStorage/ms-dotnettools.vscode-dotnet-runtime/.dotnet/9.0.11~arm64:$PATH"
  }
}
JSON_SNIPPET

# 3) Verify (run these in a new integrated terminal after restarting VS Code):
echo "After restart, run inside VS Code integrated terminal:"
echo "which dotnet && dotnet --info"

## Git: commit and push the modified notebook

Your `git status` shows the notebook is modified and unstaged. Run the commands below from the repository root to push the change.

- Stage the file: git add notebooks/C22716399_Aaron_Baggot_Assignment_1.ipynb
- Commit: git commit -m "Update notebook"
- Push: git push origin main

If push is rejected because the remote changed, run:
- git pull --rebase origin main
- resolve conflicts if any, then git push origin main

If you want to avoid large diffs, clear notebook outputs before staging (e.g. use `nbstripout` or Jupyter UI).

In [ ]:
# Run these in a shell (or prefix with ! to run inside this notebook):
# 1) Stage the modified notebook
!git add notebooks/C22716399_Aaron_Baggot_Assignment_1.ipynb
# 2) Commit (replace message as needed)
!git commit -m "Update notebook" || echo 'Nothing to commit or commit failed'
# 3) Push to main
!git push origin main || echo 'Push failed — try git pull --rebase origin main and resolve conflicts, then push'

# Optional: clear outputs before committing (uncomment to run if nbconvert is available):
# jupyter nbconvert --clear-output --inplace notebooks/C22716399_Aaron_Baggot_Assignment_1.ipynb